# 03 · Embed — 01 offline embeddings (hash fallback)

**These vectors are deterministic hashes, not semantically meaningful — this proves the wiring works, get a real key to move to `02-openai-embeddings.ipynb`.**

No model, no API key, and no network call happens anywhere in this notebook. Every vector below comes from hashing whitespace tokens with SHA-256 and accumulating them into a fixed-size vector — a deterministic fallback used whenever no embedding key is configured (`hash_embed()` below). It's the reason this repo can promise "run in 60 seconds, no key": a contributor clones, runs this notebook, and watches the whole pipeline move end to end before they have an account anywhere.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `hash_embed` | Hashes one string into a deterministic, L2-normalized vector at a given dimension — no model, no network. | `hash_embed(text, dim=384)` → a 384-dim unit vector |
| `hash_embed_batch` | Applies `hash_embed` across a list of texts. | `hash_embed_batch(sample_chunks, dim=384)` → 5 vectors |


## Step 1 — bootstrap the repo path and confirm the environment

Jupyter starts this kernel with the notebook's own directory as `cwd`, so `nbio` has to be located and put on `sys.path` before anything else can import it.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 2 — define `hash_embed`, the offline embedding function

`hash_embed()` takes `dim` as a plain argument so the dimension is explicit at the call site rather than hidden behind a settings object. The algorithm: each token is hashed with SHA-256, the hash picks one of `dim` buckets and a sign, and the accumulated vector is L2-normalized. Same text in → same vector out, always — there is no model to be unavailable and nothing to call over the network.

In [ ]:
import hashlib
import math


def hash_embed(text: str, dim: int = 384) -> list[float]:
    """Deterministic, offline embedding -- no model, no API key, no network."""
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]

## Step 3 — test `hash_embed` on one string

Look at real output from a single call before wiring it up over a whole batch: the dimension it returns and the norm of the vector it produces.

In [ ]:
DIM = 384  # a common multi-provider default dimension
test_text = "The mitochondria is the powerhouse of the cell, converting nutrients into ATP through oxidative phosphorylation."

vec = hash_embed(test_text, dim=DIM)
print(f"dim={len(vec)}  norm={math.sqrt(sum(v * v for v in vec)):.4f}")

## Step 4 — define `hash_embed_batch`

A thin wrapper that applies `hash_embed` to a list of texts — the shape a set of chunks would actually arrive in.

In [ ]:
def hash_embed_batch(texts: list[str], dim: int = 384) -> list[list[float]]:
    return [hash_embed(t, dim) for t in texts]

## Step 5 — run it on the full sample set and look at real output

These five sentences stand in for chunks that would arrive from `02-chunk`. They're hand-written for this notebook, not pulled from any external dataset.

In [ ]:
sample_chunks = [
    "The mitochondria is the powerhouse of the cell, converting nutrients into ATP through oxidative phosphorylation.",
    "Retrieval-augmented generation grounds a language model's answer in retrieved passages rather than parametric memory alone.",
    "A vector embedding maps a passage of text onto a point in a high-dimensional space so that semantic similarity becomes geometric distance.",
    "Photosynthesis converts light energy into chemical energy stored in glucose, releasing oxygen as a byproduct.",
    "An index manifest records the model, dimension and tokenizer used to build a vector store, so a mismatch is caught before it becomes a silent empty result.",
]

vectors = hash_embed_batch(sample_chunks, dim=DIM)

print(f"{len(vectors)} vectors, dimension {len(vectors[0])}")
for chunk, vec in zip(sample_chunks, vectors):
    norm = math.sqrt(sum(v * v for v in vec))
    print(f"  dim={len(vec):4d}  norm={norm:.4f}  {chunk[:60]!r}")

## Step 6 — prove it's deterministic

The whole offline promise rests on this: the same chunk hashes to the same vector every single time, with no model state and no randomness anywhere in the call.

In [ ]:
first = hash_embed(sample_chunks[0], dim=DIM)
again = hash_embed(sample_chunks[0], dim=DIM)
assert first == again, "same text must hash to the same vector, every time"
print("same input -> identical vector, confirmed (no randomness, no model, no network)")

## Next

These vectors are real wiring and fake semantics — they will not retrieve the right chunk for a query about a topic, because a hash has no notion of meaning. That's the whole point: the pipeline moved end to end with nothing installed and no account anywhere.

Set `OPENAI_API_KEY` and open `02-openai-embeddings.ipynb` for the real `text-embedding-3-large` path at 3072 dimensions.